# 认识你的 GPU：nvidia-smi 与 Tesla T4

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/rootsongjc/gpu-lab-notebooks/blob/main/labs/zh/lab-01-meet-your-gpu.ipynb)

全书第一个实验。在 Colab 免费 T4 上跑通 nvidia-smi，逐字段读懂 Driver、CUDA Version、显存、功耗、利用率，并搞清楚为什么有时会报 command not found。

本 notebook 配合教程章节 [认识你的 GPU：nvidia-smi 与 Tesla T4](https://jimmysong.io/zh/book/gpu-lab-on-kubernetes/colab/lab-01-meet-your-gpu/) 使用。运行前请确认 **Runtime → Change runtime type → T4 GPU**。

每个 cell 对应教程中的一步；Markdown 单元摘录了必要的解释，完整上下文请回到教程阅读。

In [ ]:
!nvidia-smi --query-gpu=name,driver_version,memory.total,compute_cap --format=csv

## 为什么这个问题重要

`nvidia-smi` 是 GPU 的“仪表盘”。后面所有实验都要靠它来观察：显存去哪了（[Lab 04](https://jimmysong.io/zh/book/gpu-lab-on-kubernetes/colab/lab-04-where-did-vram-go/)）、利用率（[Lab 05](https://jimmysong.io/zh/book/gpu-lab-on-kubernetes/colab/lab-05-gpu-utilization/)）、NVML（[Lab 06](https://jimmysong.io/zh/book/gpu-lab-on-kubernetes/colab/lab-06-nvml/)）。但很多人第一次在 Colab 里跑 `nvidia-smi` 就卡住：要么 `command not found`，要么拿到一块不是 GPU 的机器。本 Lab 把这两个最常见的坑一次性讲透。

## 第 0 步：确认你的 Runtime 类型

在菜单里：

```text
Runtime → Change runtime type → Hardware accelerator
```

确认显示的是 **T4 GPU**（免费版最常见），而不是 **CPU**。

> 这一步看似啰嗦，却是“看不到 GPU”的最高频原因。详见本 Lab 末尾的 [故障模式](#故障模式)。

## 第 1 步：第一次 nvidia-smi

新建一个 Notebook cell，输入：

In [ ]:
!nvidia-smi

> **行首的 ! 不能少**
>
> Colab 的 cell 默认是 Python。`!` 告诉 Colab 把这一行交给 shell 执行。漏掉 `!` 会得到 `NameError: name 'nvidia' is not defined` 或 `SyntaxError`，这不是 GPU 的问题。

预期输出类似（驱动与 CUDA 版本号会随 Google 镜像更新，以你实际看到的为准）：

```text
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 550.54.15              Driver Version: 550.54.15      CUDA Version: 12.4     |
|-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   38C    P8              9W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+------------------------+----------------------+

+-----------------------------------------------------------------------------------------+
| Processes:                                                                              |
|  GPU   GI   CI        PID   Type   Process name                              GPU Memory |
|        ID   ID                                                               Usage      |
|=========================================================================================|
|  No running processes found                                                             |
+-----------------------------------------------------------------------------------------+
```

## 第 2 步：逐字段读懂

| 字段 | 含义 | 你在 T4 上会看到 |
| :--- | :--- | :--- |
| `NVIDIA-SMI` | nvidia-smi 自身版本，与驱动同版本 | 5xx.xx |
| `Driver Version` | 内核态驱动版本 | 5xx.xx |
| `CUDA Version` | **此驱动支持的最高 CUDA Runtime**（不是已装版本） | 12.x |
| `GPU 0  Name` | 设备型号 | `Tesla T4` |
| `Persistence-M` | 持久化模式 | `Off`（Colab 正常） |
| `Bus-Id` | PCIe 总线地址 | `00000000:00:04.0`（[Lab 02](https://jimmysong.io/zh/book/gpu-lab-on-kubernetes/colab/lab-02-gpu-on-pcie/) 会用到） |
| `Disp.A` | 是否接显示器输出 | `Off`（数据中心卡没有显示接口） |
| `Perf` | 性能状态 P0（最高）到 P8/P12（最省电） | 空载 `P8`，跑起来变 `P0` |
| `Temp` | 温度 | ~35–45°C 空载 |
| `Pwr:Usage/Cap` | 当前功耗 / 功耗墙 | 空载约 `9W / 70W` |
| `Memory-Usage` | 显存用量 / 总量 | `0MiB / 15360MiB`（Colab 报告 15 GiB，不是营销口径的 16 GB） |
| `GPU-Util` | GPU 利用率 | `0%`（空载，[Lab 05](https://jimmysong.io/zh/book/gpu-lab-on-kubernetes/colab/lab-05-gpu-utilization/) 专门讲它） |
| `Compute M.` | 计算模式 | `Default`（允许多进程共用，[Lab 09](https://jimmysong.io/zh/book/gpu-lab-on-kubernetes/colab/lab-09-mps/) 会讲） |
| `MIG M.` | MIG 模式 | `N/A`（T4 不支持 MIG，[Lab 10](https://jimmysong.io/zh/book/gpu-lab-on-kubernetes/colab/lab-10-gpu-virtualization/) 会讲） |
| `Processes` | 当前用 GPU 的进程 | 空载为空；Colab 容器里即使有进程也常显示不出 PID |

> **Driver Version 旁边的 CUDA Version 是什么**
>
> `CUDA Version: 12.4` 的意思是“**这块驱动最多能支持到 CUDA 12.4 的 Runtime**”，不是“你机器上装了 CUDA 12.4”。Driver / Runtime / Toolkit 三者的区别是 [Lab 03](https://jimmysong.io/zh/book/gpu-lab-on-kubernetes/colab/lab-03-cuda-exists/) 的主题。

## 第 3 步：这块 T4 到底是什么水平

Tesla T4 是 NVIDIA 2018 年的数据中心推理卡，今天仍在大量生产环境服役：

| 参数 | Tesla T4 |
| :--- | :--- |
| 架构 | Turing（TU104），Compute Capability 7.5 |
| SM 数量 | 40 |
| CUDA Cores | 2560（每个 SM 64 个） |
| Tensor Cores | 320（第二代，支持 FP16 / INT8 / INT4） |
| FP32 峰值 | 约 8.1 TFLOPS |
| 显存 | 16 GB GDDR6（`nvidia-smi` 报告 15360 MiB） |
| 显存带宽 | 320 GB/s |
| 功耗墙 | 70 W（无风扇，被动散热） |
| 接口 | PCIe Gen3 x16 |

放到 2026 年，它不是高性能卡，但作为学 CUDA / 深度学习 / 推理的**入门平台**非常合适：它是 CUDA 13 仍支持的最老架构（Compute Capability 7.5，[Lab 07](https://jimmysong.io/zh/book/gpu-lab-on-kubernetes/colab/lab-07-cuda-programming-model/) 会解释这个数字），覆盖了后面所有 Lab 需要的能力。

## 验证

完成本 Lab 后，你应该能：

- 解释 `Driver Version` 与 `CUDA Version` 的区别；
- 说出 T4 的显存总量、功耗墙、PCIe 地址；
- 知道 `Memory-Usage` 和 `GPU-Util` 分别在哪一行。

## 故障模式

| 症状 | 原因 | 处理 |
| :--- | :--- | :--- |
| `/bin/bash: nvidia-smi: command not found` | Runtime 是 **CPU**，不是 GPU | Runtime → Change runtime type → 选 **T4 GPU**，重新连接 |
| `NameError: name 'nvidia' is not defined` | 漏了行首的 `!`，Colab 把它当 Python 执行 | 改成 `!nvidia-smi` |
| `nvdia-smi: command not found`（拼写错） | 你把命令打错了 | 注意是 `nvidia-smi`，不是 `nvdia-smi` |
| 切换 runtime 后提示“无法连接到 GPU 后端” | 免费配额用尽或 GPU 资源紧张 | 稍后再试，或先在 CPU runtime 上阅读本章 |
| 重新连接后显存不是 0 MiB | 本 notebook 里之前的 Python 进程仍在占用 | Runtime → Restart session；不要尝试 `nvidia-smi --gpu-reset`，Colab 容器没有这个权限 |
| `NVIDIA-SMI has failed because it couldn't communicate with the NVIDIA driver` | Runtime 刚切换尚未就绪 | 等 10–30 秒重跑 |

## 跑完之后

如果你看到的输出与教程里的预期明显不符，或者碰到了教程没有列出的故障模式，欢迎带上驱动 / CUDA 版本和关键输出在 [rootsongjc/gpu-lab-notebooks](https://github.com/rootsongjc/gpu-lab-notebooks) 提 Issue 或 PR。